In [1]:
import numpy as np
import pandas as pd
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, f1_score
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE, ADASYN, RandomOverSampler
import matplotlib.pyplot as plt
import warnings
from alive_progress import alive_bar
from contextlib import contextmanager
from joblib.parallel import BatchCompletionCallBack
import threading
from math import prod
from sklearn.exceptions import ConvergenceWarning, UndefinedMetricWarning
import logging

import os
os.environ['PYTHONWARNINGS'] = "ignore"

# Suppress warnings for cleaner output
warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore", category=UndefinedMetricWarning)

# Suppress logging warnings by setting the logging level to ERROR
logging.getLogger().setLevel(logging.ERROR)

# Load the dataset
# Replace the file path with your actual path
data = pd.read_excel("class123_dataset.xlsx")

# Extract the predictors and outcome
X = data.drop('RRI', axis=1)
Y = data['RRI']

# Define the feature indexes to be used as predictors
# Note: Pandas uses 0-based indexing
feature_indexes = [138, 244, 224, 137, 118, 103, 204, 230, 3, 86, 42, 24, 13, 21, 78, 33, 88, 183, 123, 119, 126, 210, 122, 65, 201, 5, 253, 30, 109, 227, 90, 49, 214, 52, 80, 25, 247, 217, 22, 229, 157, 115]

# Select the specified features using .iloc
X_selected = X.iloc[:, feature_indexes]

# Initialize the Support Vector Classifier with probability estimates enabled
svm_classifier = SVC(probability=True, random_state=42)

# Create the pipeline with sampler and classifier
pipeline = ImbPipeline([
    ('sampler', RandomOverSampler()),  # Placeholder, will be set in GridSearchCV
    ('classifier', svm_classifier)
])

# Define the parameter grid for hyperparameter tuning
param_grid = [
        # No sampling strategy
    {
        'sampler': [None],
        'classifier__kernel': ['linear'],
        'classifier__C': [10, 50, 100],
        'classifier__class_weight': [None, 'balanced'],
        'classifier__tol': [1e-3, 1e-5],
        'classifier__max_iter': [20000]  # Additional max_iter value for linear kernel
    },
    {
        'sampler': [None],
        'classifier__kernel': ['rbf'],
        'classifier__C': [10, 50, 100],
        'classifier__gamma': ['scale', 'auto', 1e-3],
        'classifier__class_weight': [None, 'balanced'],
        'classifier__tol': [1e-3, 1e-5],
        'classifier__max_iter': [20000]  # Additional max_iter value for rbf kernel
    },
    {
        'sampler': [None],
        'classifier__kernel': ['poly'],
        'classifier__C': [10, 50, 100],
        'classifier__gamma': ['scale', 'auto', 1e-3],
        'classifier__degree': [2, 4, 5],
        'classifier__coef0': [0.0, 0.5, 1.0],
        'classifier__class_weight': [None, 'balanced'],
        'classifier__tol': [1e-3, 1e-5],
        'classifier__max_iter': [20000]  # Additional max_iter value for poly kernel
    },
    {
        'sampler': [RandomOverSampler(), SMOTE(), ADASYN()],
        'sampler__sampling_strategy': ['auto', 0.25, 0.5, 1.0],
        'classifier__kernel': ['linear'],
        'classifier__C': [10, 50, 100],
        'classifier__class_weight': [None, 'balanced'],
        'classifier__tol': [1e-3, 1e-5],
        'classifier__max_iter': [20000]  # Additional max_iter value for linear kernel
    },
    {
        'sampler': [RandomOverSampler(), SMOTE(), ADASYN()],
        'sampler__sampling_strategy': ['auto', 0.25, 0.5, 1.0],
        'classifier__kernel': ['rbf'],
        'classifier__C': [10, 50, 100],
        'classifier__gamma': ['scale', 'auto', 1e-3],
        'classifier__class_weight': [None, 'balanced'],
        'classifier__tol': [1e-3, 1e-5],
        'classifier__max_iter': [20000]  # Additional max_iter value for rbf kernel
    },
    {
        'sampler': [RandomOverSampler(), SMOTE(), ADASYN()],
        'sampler__sampling_strategy': ['auto', 0.25, 0.5, 1.0],
        'classifier__kernel': ['poly'],
        'classifier__C': [10, 50, 100],
        'classifier__gamma': ['scale', 'auto', 1e-3],
        'classifier__degree': [2, 4, 5],
        'classifier__coef0': [0.0, 0.5, 1.0],
        'classifier__class_weight': [None, 'balanced'],
        'classifier__tol': [1e-3, 1e-5],
        'classifier__max_iter': [20000]  # Additional max_iter value for poly kernel
    }
]

# Define the scoring metrics
scoring = {
    'roc_auc': 'roc_auc',
    'accuracy': 'accuracy',
    'precision': 'precision',
    'f1': 'f1'
}

# Define the Stratified 10-Fold Cross-Validation
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# Calculate the total number of parameter combinations for the progress bar
total_fits = int(
    sum(
        np.prod([len(values) for values in grid.values()])
        for grid in param_grid
    )
)

@contextmanager
def alive_joblib_bar(total):
    """
    Context manager to integrate alive-progress with joblib's Parallel processing.

    Parameters:
    - total: int, the total number of tasks to be processed.
    """
    with alive_bar(total, title='Grid Search Progress', bar='blocks', force_tty=True) as bar:
        # Store the original BatchCompletionCallBack.__call__ method
        original_callback = BatchCompletionCallBack.__call__
        lock = threading.Lock()

        def on_complete(self, *args, **kwargs):
            with lock:
                bar()
            return original_callback(self, *args, **kwargs)

        # Patch the BatchCompletionCallBack.__call__ method
        BatchCompletionCallBack.__call__ = on_complete
        try:
            yield
        finally:
            # Restore the original method to avoid side effects
            BatchCompletionCallBack.__call__ = original_callback

# Initialize Grid Search with cross-validation
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring=scoring,
    refit='roc_auc',  # Use 'roc_auc' to select the best model
    cv=cv,
    n_jobs=-1,  # Use all available cores
    verbose=0,  # Disable sklearn's verbose
    return_train_score=False
)

# Start the Grid Search with alive-progress
print("Starting Grid Search...")

with warnings.catch_warnings():
    warnings.filterwarnings("ignore", category=ConvergenceWarning)
    warnings.filterwarnings("ignore", category=UndefinedMetricWarning)
    with alive_joblib_bar(total_fits):
        grid_search.fit(X_selected, Y)

print("Grid Search Completed.")

# Retrieve the best average AUC and its standard deviation
best_auc = grid_search.best_score_
# Retrieve the standard deviation from cv_results_
# Identify the index of the best parameter set
best_index = grid_search.best_index_
best_auc_std = grid_search.cv_results_['std_test_roc_auc'][best_index]

# Retrieve the best hyperparameters
best_params = grid_search.best_params_

# Retrieve the other metrics for the best parameter set
best_accuracy = grid_search.cv_results_['mean_test_accuracy'][best_index]
best_accuracy_std = grid_search.cv_results_['std_test_accuracy'][best_index]

best_precision = grid_search.cv_results_['mean_test_precision'][best_index]
best_precision_std = grid_search.cv_results_['std_test_precision'][best_index]

best_f1 = grid_search.cv_results_['mean_test_f1'][best_index]
best_f1_std = grid_search.cv_results_['std_test_f1'][best_index]

# Output the results
print(f"\nBest Average AUC: {best_auc:.4f} ± {best_auc_std:.4f}")
print(f"Average Accuracy: {best_accuracy:.4f} ± {best_accuracy_std:.4f}")
print(f"Average Precision: {best_precision:.4f} ± {best_precision_std:.4f}")
print(f"Average F1 Score: {best_f1:.4f} ± {best_f1_std:.4f}")
print("\nBest Hyperparameters:")
for param, value in best_params.items():
    if isinstance(value, (SMOTE, ADASYN, RandomOverSampler)):
        print(f"  {param}: {value.__class__.__name__}")
    else:
        print(f"  {param}: {value}")

Starting Grid Search...
                                                                                [rid Search Progress |                                        | ▇▇▅ 0/4836 [0%] Grid Search Progress |                                        | ▂▄▆ 1/4836 [0%] Grid Search Progress |▏                                       | ▆▄▂ 10/4836 [0%]Grid Search Progress |▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉✗︎ ▁▃▅ 5858/4836 [1Grid Search Progress |▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉✗︎ ▄▆█ 5859/4836 [1Grid Search Progress |▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉✗︎ ▇▅▃ 5916/4836 [1Grid Search Progress |▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉✗︎ ▇▇▅ 5985/4836 [1Grid Search Progress |▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉✗︎ ▅▃▁ 5990/4836 [1Grid Search Progress |▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉✗︎ ▇▅▃ 6001/4836 [1Grid Search Progress |▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉✗︎ ▃▅▇ 6008/4836 [1Grid Search Progress |▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉✗︎ ▂▂▄ 6013/4836 [1Grid Sea

on 48360: /home/azureuser/myenv/lib/python3.12/site-packages/numpy/ma/core.py:2881: RuntimeWarning: invalid value encountered in cast
            _data = np.array(data, dtype=dtype, copy=copy,


Grid Search Progress |▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉▉✗︎ (!) 48360/4836 [
Grid Search Completed.

Best Average AUC: 0.7500 ± 0.0193
Average Accuracy: 0.8882 ± 0.0100
Average Precision: 0.3663 ± 0.0599
Average F1 Score: 0.3294 ± 0.0515

Best Hyperparameters:
  classifier__C: 10
  classifier__class_weight: None
  classifier__coef0: 0.5
  classifier__degree: 5
  classifier__gamma: scale
  classifier__kernel: poly
  classifier__max_iter: 20000
  classifier__tol: 1e-05
  sampler: RandomOverSampler
  sampler__sampling_strategy: 0.25


In [3]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_predict

# Retrieve the best estimator from Grid Search
best_estimator = grid_search.best_estimator_

# Use cross_val_predict to get cross-validated predicted probabilities
# Setting method='predict_proba' and using cv to ensure consistency
print("\nGenerating cross-validated predicted probabilities...")
y_pred_proba = cross_val_predict(best_estimator, X_selected, Y, cv=cv, method='predict_proba', n_jobs=-1)[:, 1]

# Define a range of threshold values to evaluate
thresholds = np.linspace(0.0, 1.0, 101)

# Initialize variables to store the best metrics and threshold
best_threshold = 0.5
best_f1_score = 0.0
best_accuracy = 0.0
best_precision = 0.0

print("Optimizing threshold to maximize F1 score...")

for threshold in thresholds:
    # Convert predicted probabilities to binary predictions based on the threshold
    y_pred = (y_pred_proba >= threshold).astype(int)
    
    # Calculate F1 score
    current_f1 = f1_score(Y, y_pred)
    
    # Update the best metrics and threshold if current F1 is better
    if current_f1 > best_f1_score:
        best_f1_score = current_f1
        best_threshold = threshold
        best_accuracy = accuracy_score(Y, y_pred)
        best_precision = precision_score(Y, y_pred, zero_division=0)

# Calculate standard deviations using cross-validation
# To compute standard deviations, we'll perform cross-validation predictions and calculate metrics at the best threshold

# Initialize lists to store per-fold metrics
f1_scores = []
accuracies = []
precisions = []

print("\nCalculating metrics at the optimal threshold across folds...")

for fold, (train_idx, test_idx) in enumerate(cv.split(X_selected, Y), 1):
    # Split data
    X_train, X_test = X_selected.iloc[train_idx], X_selected.iloc[test_idx]
    y_train, y_test = Y.iloc[train_idx], Y.iloc[test_idx]
    
    # Fit the model on the training data
    best_estimator.fit(X_train, y_train)
    
    # Predict probabilities on the test data
    y_proba_fold = best_estimator.predict_proba(X_test)[:, 1]
    
    # Apply the optimal threshold
    y_pred_fold = (y_proba_fold >= best_threshold).astype(int)
    
    # Calculate metrics
    fold_f1 = f1_score(y_test, y_pred_fold)
    fold_accuracy = accuracy_score(y_test, y_pred_fold)
    fold_precision = precision_score(y_test, y_pred_fold, zero_division=0)
    
    # Append to lists
    f1_scores.append(fold_f1)
    accuracies.append(fold_accuracy)
    precisions.append(fold_precision)
    
    print(f"  Fold {fold}: F1={fold_f1:.4f}, Accuracy={fold_accuracy:.4f}, Precision={fold_precision:.4f}")

# Calculate mean and standard deviation for the metrics
mean_f1 = np.mean(f1_scores)
std_f1 = np.std(f1_scores)

mean_accuracy = np.mean(accuracies)
std_accuracy = np.std(accuracies)

mean_precision = np.mean(precisions)
std_precision = np.std(precisions)

# Output the optimized threshold and corresponding metrics
print(f"\n=== Optimized Threshold ===")
print(f"Threshold for Maximum F1 Score: {best_threshold:.2f}")

print(f"\n=== Metrics at Optimal Threshold ===")
print(f"F1 Score: {mean_f1:.4f} ± {std_f1:.4f}")
print(f"Accuracy: {mean_accuracy:.4f} ± {std_accuracy:.4f}")
print(f"Precision: {mean_precision:.4f} ± {std_precision:.4f}")


Generating cross-validated predicted probabilities...
Optimizing threshold to maximize F1 score...

Calculating metrics at the optimal threshold across folds...
  Fold 1: F1=0.3906, Accuracy=0.8740, Precision=0.3521
  Fold 2: F1=0.3455, Accuracy=0.8835, Precision=0.3519
  Fold 3: F1=0.3967, Accuracy=0.8819, Precision=0.3692
  Fold 4: F1=0.3200, Accuracy=0.8625, Precision=0.2899
  Fold 5: F1=0.3651, Accuracy=0.8706, Precision=0.3286
  Fold 6: F1=0.2787, Accuracy=0.8576, Precision=0.2576
  Fold 7: F1=0.3784, Accuracy=0.8883, Precision=0.3818
  Fold 8: F1=0.3594, Accuracy=0.8673, Precision=0.3239
  Fold 9: F1=0.3361, Accuracy=0.8722, Precision=0.3226
  Fold 10: F1=0.4324, Accuracy=0.8981, Precision=0.4444

=== Optimized Threshold ===
Threshold for Maximum F1 Score: 0.31

=== Metrics at Optimal Threshold ===
F1 Score: 0.3603 ± 0.0411
Accuracy: 0.8756 ± 0.0117
Precision: 0.3422 ± 0.0487
